# Notebook 12 — Condition Analysis Training (Reviewed and Corrected for Final 15K Dataset)

**Project:** Adaptive Road Vision Enhancement and Risk-Aware Object Detection under Day, Night, Rain and Fog Conditions  
**Project root:** detected from the notebook/project folders; fallback is `C:\\Users\\ankit\\Desktop\\Adaptive_Road_Vision`.  
**Input:** the final 15,000-image dataset created by Notebook 10 and checked by Notebook 11.  
**Purpose:** train a five-class image-level condition classifier (`Day`, `Night`, `Rain`, `Fog`, `Unknown`). This is a condition classifier, not the YOLO object detector. YOLO labels are not used as classifier targets.

Expected split counts: **10,500 train / 1,500 validation / 3,000 test**. Synthetic Fog images must be training-only. Test data is used only for final evaluation.


In [ ]:
from pathlib import Path
import json
import random
import time
import copy
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Project root for this project on the user's Windows machine.
# Change only this line if your project is stored elsewhere.
def find_project_root():
    """Find the project root without assuming a specific Windows username."""
    current = Path.cwd().resolve()
    candidates = [current, *list(current.parents[:5])]
    for candidate in candidates:
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate

    # Known project location from the earlier project setup.
    fallback = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
    if fallback.exists():
        return fallback

    raise FileNotFoundError(
        "Could not detect Adaptive_Road_Vision project root. "
        "Open this notebook from the project folder, or set PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root()

# Final 15K dataset folder created by Notebook 10.
DATASET_DIR_CANDIDATES = [
    PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
    PROJECT_ROOT / "datasets" / "final_condition_balanced",
    PROJECT_ROOT / "datasets" / "final_balanced_15k",
    PROJECT_ROOT / "datasets" / "final_15k",
    PROJECT_ROOT / "datasets" / "final",
]

# Metadata names used across versions of Notebooks 10 and 11.
METADATA_NAMES = [
    "final_dataset_metadata.csv",
    "dataset_metadata.csv",
    "final_metadata.csv",
    "final_dataset.csv",
    "balanced_dataset_metadata.csv",
    "dataset_balance_metadata.csv",
    "final_metadata.csv",
    "dataset_manifest.csv",
]
metadata_candidates = []
for dataset_dir in DATASET_DIR_CANDIDATES:
    for name in METADATA_NAMES:
        metadata_candidates.extend([
            dataset_dir / "metadata" / name,
            dataset_dir / "reports" / name,
            dataset_dir / name,
        ])

FINAL_METADATA_CSV = next((p for p in metadata_candidates if p.is_file()), None)
if FINAL_METADATA_CSV is None:
    # Search only inside the configured final-dataset folders, not the entire project,
    # to avoid accidentally selecting a source BDD100K metadata file.
    for dataset_dir in DATASET_DIR_CANDIDATES:
        if dataset_dir.exists():
            found = sorted(
                p for p in dataset_dir.rglob("*.csv")
                if any(token in p.name.lower() for token in ("metadata", "manifest", "final_dataset"))
            )
            if found:
                FINAL_METADATA_CSV = found[0]
                break

MODEL_DIR = PROJECT_ROOT / "models" / "condition"
RESULTS_DIR = PROJECT_ROOT / "results" / "condition"
LOG_DIR = PROJECT_ROOT / "logs" / "condition"
for folder in (MODEL_DIR, RESULTS_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

BEST_MODEL = MODEL_DIR / "condition_resnet18_best.pth"
LAST_MODEL = MODEL_DIR / "condition_resnet18_last.pth"
HISTORY_JSON = RESULTS_DIR / "training_history.json"
METRICS_JSON = RESULTS_DIR / "condition_training_metrics.json"
CONFUSION_CSV = RESULTS_DIR / "condition_confusion_matrix.csv"
INCORRECT_CSV = RESULTS_DIR / "incorrect_test_predictions.csv"

SEED = 42
EXPECTED_TOTAL = 15000
EXPECTED_SPLITS = {"train": 10500, "test": 3000, "val": 1500}
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PATIENCE = 5

CLASS_NAMES = ["Day", "Night", "Rain", "Fog", "Unknown"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {i: name for name, i in CLASS_TO_ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project:", PROJECT_ROOT)
print("Dataset metadata:", FINAL_METADATA_CSV)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root does not exist: {PROJECT_ROOT}")
if FINAL_METADATA_CSV is None or not FINAL_METADATA_CSV.is_file():
    raise FileNotFoundError(
        "Could not find final dataset metadata. Expected final_dataset_metadata.csv "
        "under datasets/final_condition_balanced_v2/ (or the other configured final dataset folders)."
    )


## 1. Verify inputs from Notebook 10/11


In [ ]:
metadata = pd.read_csv(FINAL_METADATA_CSV)

# Normalize column names from Notebook 10/11 variants.
if "project_condition" not in metadata.columns and "condition" in metadata.columns:
    metadata["project_condition"] = metadata["condition"]
if "final_image_path" not in metadata.columns and "image_path" in metadata.columns:
    metadata["final_image_path"] = metadata["image_path"]
if "image_id" not in metadata.columns:
    if "source_id" in metadata.columns:
        metadata["image_id"] = metadata["source_id"].astype(str)
    elif "image_name" in metadata.columns:
        metadata["image_id"] = metadata["image_name"].astype(str)
    elif "final_image_path" in metadata.columns:
        metadata["image_id"] = metadata["final_image_path"].astype(str)
    elif "image_path" in metadata.columns:
        metadata["image_id"] = metadata["image_path"].astype(str)
    else:
        metadata["image_id"] = metadata.index.astype(str)

if "image_name" not in metadata.columns:
    path_column = "final_image_path" if "final_image_path" in metadata.columns else "image_path"
    if path_column not in metadata.columns:
        raise ValueError("Metadata needs 'final_image_path' or 'image_path'.")
    metadata["image_name"] = metadata[path_column].map(lambda p: Path(str(p)).name)

# Normalize synthetic values safely (bool("False") is True, so do not use bool() directly).
def normalize_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return False
    if isinstance(value, (int, np.integer, float, np.floating)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}

if "synthetic" not in metadata.columns:
    if "record_type" in metadata.columns:
        metadata["synthetic"] = metadata["record_type"].astype(str).str.lower().str.contains("synthetic")
    else:
        raise ValueError("Metadata must contain either 'synthetic' or 'record_type'.")
metadata["synthetic"] = metadata["synthetic"].map(normalize_bool)

if "record_type" not in metadata.columns:
    metadata["record_type"] = np.where(metadata["synthetic"], "synthetic_fog", "original")
else:
    metadata["record_type"] = np.where(metadata["synthetic"], "synthetic_fog", "original")

def resolve_img_path(value):
    raw = str(value).strip()
    p = Path(raw)
    if p.is_file():
        return str(p.resolve())
    if not p.is_absolute():
        candidate = PROJECT_ROOT / p
        if candidate.is_file():
            return str(candidate.resolve())
        candidate = FINAL_METADATA_CSV.parent / p
        if candidate.is_file():
            return str(candidate.resolve())
        # Handle paths stored relative to the dataset root.
        for dataset_dir in DATASET_DIR_CANDIDATES:
            candidate = dataset_dir / p
            if candidate.is_file():
                return str(candidate.resolve())
    return str(p)

required = ["record_type", "image_id", "image_name", "split", "project_condition", "final_image_path", "synthetic"]
missing = [c for c in required if c not in metadata.columns]
if missing:
    raise ValueError(f"Missing metadata columns: {missing}")

metadata["split"] = metadata["split"].astype(str).str.strip().str.lower()
if "source_split" in metadata.columns:
    metadata["source_split"] = metadata["source_split"].astype(str).str.strip().str.lower()
metadata["project_condition"] = metadata["project_condition"].fillna("Unknown").astype(str).str.strip().str.title()
metadata.loc[metadata["project_condition"].isin(["", "Nan", "None", "Na"]), "project_condition"] = "Unknown"
condition_aliases = {
    "Foggy": "Fog",
    "Rainy": "Rain",
    "Daytime": "Day",
    "Nighttime": "Night",
    "Na": "Unknown",
    "Nan": "Unknown",
    "None": "Unknown",
}
metadata["project_condition"] = metadata["project_condition"].replace(condition_aliases)
metadata["final_image_path"] = metadata["final_image_path"].map(resolve_img_path)

# Accept only the expected three split names; catch common spelling errors early.
unexpected_splits = sorted(set(metadata["split"]) - {"train", "val", "test"})
if unexpected_splits:
    raise ValueError(f"Unexpected split names: {unexpected_splits}. Expected train/val/test.")

if metadata["final_image_path"].duplicated().any():
    duplicate_count = int(metadata["final_image_path"].duplicated().sum())
    raise ValueError(f"Duplicate image paths in metadata: {duplicate_count}")

print("Metadata file:", FINAL_METADATA_CSV)
print("Metadata shape:", metadata.shape)
print("\\nSplit counts:")
print(metadata["split"].value_counts().reindex(["train", "val", "test"], fill_value=0))
print("\\nCondition counts:")
print(metadata["project_condition"].value_counts())
print("\\nSynthetic counts:")
print(metadata["synthetic"].value_counts(dropna=False))


In [ ]:
# Verify final dataset size and split policy.
assert len(metadata) == EXPECTED_TOTAL, (
    f"Expected {EXPECTED_TOTAL:,} metadata rows, found {len(metadata):,}. "
    "Check that Notebook 10's final dataset metadata is being used."
)

actual_splits = metadata["split"].value_counts().to_dict()
assert actual_splits == EXPECTED_SPLITS, (
    f"Expected split counts {EXPECTED_SPLITS}, found {actual_splits}"
)

original = metadata[~metadata["synthetic"]].copy()
synthetic = metadata[metadata["synthetic"]].copy()

assert len(original[original["split"] == "train"]) > 0, "No original training images!"
assert len(original[original["split"] == "val"]) > 0, "No original validation images!"
assert len(original[original["split"] == "test"]) > 0, "No original test images!"
assert synthetic.empty or synthetic["split"].eq("train").all(), "Synthetic images must be train only!"

# If source_split exists, original images must not cross their original split.
if "source_split" in original.columns:
    original_source_split = original["source_split"].astype(str).str.strip().str.lower()
    original_output_split = original["split"].astype(str)
    mismatches = original_source_split != original_output_split
    assert not mismatches.any(), (
        f"{int(mismatches.sum())} original records do not preserve source_split."
    )

print(f"Dataset total: {len(metadata):,}")
print(f"Original records: {len(original):,}")
print(f"Synthetic records: {len(synthetic):,}")
print("Split counts:", actual_splits)
print("Synthetic-only-train and split integrity: PASS")


## 2. Prepare condition labels

The condition labels come directly from the `project_condition` field created during the earlier BDD100K processing workflow.

No condition label is inferred from the image here.


In [ ]:
invalid_conditions = sorted(
    set(metadata["project_condition"].astype(str)) - set(CLASS_NAMES)
)

if invalid_conditions:
    raise ValueError(
        f"Unexpected condition labels found: {invalid_conditions}. Expected one of {CLASS_NAMES}. Check Notebook 10 metadata labels."
    )

metadata["label"] = metadata["project_condition"].map(CLASS_TO_ID)

print(metadata["project_condition"].value_counts())
print("\\nClass mapping:")
print(CLASS_TO_ID)


## 3. Build train/validation/test dataframes


In [ ]:
train_df = metadata[metadata["split"] == "train"].copy().reset_index(drop=True)
val_df = metadata[metadata["split"] == "val"].copy().reset_index(drop=True)
test_df = metadata[metadata["split"] == "test"].copy().reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

assert train_df["split"].eq("train").all()
assert val_df["split"].eq("val").all()
assert test_df["split"].eq("test").all()

print("\\nTrain condition distribution:")
print(train_df["project_condition"].value_counts())

print("\\nValidation condition distribution:")
print(val_df["project_condition"].value_counts())

print("\\nTest condition distribution:")
print(test_df["project_condition"].value_counts())


## 4. Check that all image paths exist


In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    exists = df["final_image_path"].map(lambda p: Path(str(p)).exists())
    print(f"{name}: {int(exists.sum())}/{len(df)} image files exist")
    assert exists.all(), f"Missing images in {name}"

print("Image-path verification: PASS")


## 5. Training transforms

Training receives moderate image augmentation.

Validation and test use deterministic preprocessing only.

No synthetic image is created here.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


## 6. PyTorch condition dataset


In [ ]:
class ConditionDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            Path(str(row["final_image_path"]))
        ).convert("RGB")

        image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(
                int(row["label"]),
                dtype=torch.long
            ),
            "condition": str(row["project_condition"]),
            "image_id": str(row["image_id"]),
            "image_name": str(row["image_name"]),
            "synthetic": bool(row["synthetic"]),
        }


train_dataset = ConditionDataset(train_df, train_transform)
val_dataset = ConditionDataset(val_df, eval_transform)
test_dataset = ConditionDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


## 7. Calculate class weights

The project has a strong class imbalance, especially among the original Fog and Unknown classes.

Class weights are calculated from the **training set only** and passed to cross-entropy loss.

This does not change validation/test distributions.


In [ ]:
train_counts = (
    train_df["label"]
    .value_counts()
    .reindex(range(len(CLASS_NAMES)), fill_value=0)
    .sort_index()
)

print("Training counts:")
for idx, count in train_counts.items():
    print(f"{idx}: {ID_TO_CLASS[idx]} = {count}")

# Inverse-frequency weighting normalized around mean weight 1.
weights = len(train_df) / (
    len(CLASS_NAMES) * train_counts.replace(0, np.nan)
)
weights = weights.fillna(0)

weights = weights / weights[weights > 0].mean()

CLASS_WEIGHTS = torch.tensor(
    weights.values,
    dtype=torch.float32,
    device=DEVICE
)

print("\\nClass weights:")
for idx, weight in enumerate(CLASS_WEIGHTS.cpu().numpy()):
    print(f"{ID_TO_CLASS[idx]}: {weight:.4f}")


## 8. Build the ResNet-18 condition classifier

A standard ResNet-18 backbone is used with a five-class classification head.

The notebook first tries ImageNet-pretrained weights. If pretrained weights are unavailable, it automatically falls back to training from scratch.


In [ ]:
try:
    weights_enum = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights_enum)
    pretrained_used = True
    print("Loaded ImageNet-pretrained ResNet-18.")
except Exception as exc:
    print("Pretrained weights unavailable:", repr(exc))
    model = models.resnet18(weights=None)
    pretrained_used = False
    print("Using ResNet-18 from scratch.")

model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model = model.to(DEVICE)

print(model.fc)


## 9. Loss, optimizer, scheduler, and AMP


In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

AMP_ENABLED = DEVICE.type == "cuda"

try:
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=AMP_ENABLED
    )
except Exception:
    scaler = torch.cuda.amp.GradScaler(
        enabled=AMP_ENABLED
    )

print("AMP enabled:", AMP_ENABLED)


## 10. Metrics

The training loop calculates:

- accuracy
- balanced accuracy
- macro F1
- per-class precision
- per-class recall
- per-class F1

The implementation uses only NumPy/Pandas so the notebook does not require scikit-learn.


In [ ]:
def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)

    for true, pred in zip(y_true, y_pred):
        cm[int(true), int(pred)] += 1

    return cm


def classification_metrics(y_true, y_pred, num_classes):
    cm = confusion_matrix_np(y_true, y_pred, num_classes)

    total = cm.sum()
    accuracy = np.trace(cm) / total if total else 0.0

    recalls = []
    precisions = []
    f1s = []

    for i in range(num_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0

        if precision + recall:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    balanced_accuracy = float(np.mean(recalls))
    macro_f1 = float(np.mean(f1s))

    return {
        "accuracy": float(accuracy),
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "precision": precisions,
        "recall": recalls,
        "f1": f1s,
        "confusion_matrix": cm,
    }


## 11. Training/evaluation functions


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    all_true = []
    all_pred = []

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        try:
            autocast_context = torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            )
        except Exception:
            autocast_context = torch.cuda.amp.autocast(
                enabled=AMP_ENABLED
            )

        with torch.set_grad_enabled(training):
            with autocast_context:
                logits = model(images)
                loss = criterion(logits, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        all_true.extend(labels.detach().cpu().numpy().tolist())
        all_pred.extend(predictions.detach().cpu().numpy().tolist())

    epoch_loss = running_loss / len(loader.dataset)

    metrics = classification_metrics(
        all_true,
        all_pred,
        len(CLASS_NAMES)
    )

    metrics["loss"] = float(epoch_loss)

    return metrics, all_true, all_pred


## 12. Train the condition classifier


In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "train_balanced_accuracy": [],
    "val_balanced_accuracy": [],
    "train_macro_f1": [],
    "val_macro_f1": [],
    "learning_rate": [],
}

best_score = -np.inf
best_state = None
epochs_without_improvement = 0

print("Starting training...")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Device: {DEVICE}")

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
    )

    scheduler.step(val_metrics["macro_f1"])

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])
    history["train_accuracy"].append(train_metrics["accuracy"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    history["train_balanced_accuracy"].append(
        train_metrics["balanced_accuracy"]
    )
    history["val_balanced_accuracy"].append(
        val_metrics["balanced_accuracy"]
    )
    history["train_macro_f1"].append(train_metrics["macro_f1"])
    history["val_macro_f1"].append(val_metrics["macro_f1"])
    history["learning_rate"].append(current_lr)

    elapsed = time.time() - start

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} | "
        f"val acc {val_metrics['accuracy']:.4f} | "
        f"val bal-acc {val_metrics['balanced_accuracy']:.4f} | "
        f"val macro-F1 {val_metrics['macro_f1']:.4f} | "
        f"lr {current_lr:.2e} | "
        f"{elapsed:.1f}s"
    )

    score = val_metrics["macro_f1"]

    if score > best_score:
        best_score = score
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": best_state,
                "class_names": CLASS_NAMES,
                "class_to_id": CLASS_TO_ID,
                "image_size": IMAGE_SIZE,
                "pretrained_used": pretrained_used,
                "epoch": epoch,
                "val_macro_f1": best_score,
            },
            BEST_MODEL,
        )

        print("  Saved best checkpoint.")
    else:
        epochs_without_improvement += 1

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "class_names": CLASS_NAMES,
            "class_to_id": CLASS_TO_ID,
            "image_size": IMAGE_SIZE,
            "epoch": epoch,
        },
        LAST_MODEL,
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {epoch} epochs.")
        break

with open(HISTORY_JSON, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

print("Training complete.")
print("Best validation macro-F1:", best_score)
print("Best checkpoint:", BEST_MODEL)


## 13. Plot training history


In [ ]:
epochs_done = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_done, history["train_loss"], label="Train loss")
plt.plot(epochs_done, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Condition classifier loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(
    epochs_done,
    history["train_macro_f1"],
    label="Train macro-F1"
)
plt.plot(
    epochs_done,
    history["val_macro_f1"],
    label="Validation macro-F1"
)
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Condition classifier macro-F1")
plt.legend()
plt.tight_layout()
plt.show()


## 14. Load the best checkpoint and evaluate on the test set

The test set is evaluated only after model selection using validation macro-F1.


In [ ]:
if not BEST_MODEL.exists():
    raise FileNotFoundError("Best checkpoint was not created.")

try:
    checkpoint = torch.load(BEST_MODEL, map_location=DEVICE, weights_only=False)
except TypeError:  # Compatibility with older PyTorch versions
    checkpoint = torch.load(BEST_MODEL, map_location=DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_metrics, test_true, test_pred = run_epoch(
    model,
    test_loader,
    criterion,
    optimizer=None,
)

print("Test loss:", test_metrics["loss"])
print("Test accuracy:", test_metrics["accuracy"])
print("Test balanced accuracy:", test_metrics["balanced_accuracy"])
print("Test macro-F1:", test_metrics["macro_f1"])


## 15. Per-class test metrics


In [ ]:
per_class = pd.DataFrame({
    "condition": CLASS_NAMES,
    "precision": test_metrics["precision"],
    "recall": test_metrics["recall"],
    "f1": test_metrics["f1"],
})

display(per_class)


## 16. Test confusion matrix


In [ ]:
cm = test_metrics["confusion_matrix"]

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

cm_df.index.name = "True"
cm_df.columns.name = "Predicted"

display(cm_df)

cm_df.to_csv(CONFUSION_CSV)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Condition classification confusion matrix")
plt.colorbar()
plt.xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45)
plt.yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")

for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.show()


## 17. Inspect incorrect test predictions

This creates a CSV containing incorrect test predictions so difficult cases can be reviewed later.


In [ ]:
test_results = test_df.copy()

test_results["true_label"] = test_results["project_condition"]
test_results["predicted_id"] = test_pred
test_results["predicted_condition"] = [
    ID_TO_CLASS[int(x)] for x in test_pred
]

test_results["correct"] = (
    test_results["true_label"]
    == test_results["predicted_condition"]
)

incorrect = test_results[~test_results["correct"]].copy()

INCORRECT_CSV = RESULTS_DIR / "incorrect_test_predictions.csv"
incorrect.to_csv(INCORRECT_CSV, index=False)

print("Incorrect test predictions:", len(incorrect))
print("Saved:", INCORRECT_CSV)

display(
    incorrect[
        [
            "image_id",
            "image_name",
            "true_label",
            "predicted_condition",
            "final_image_path",
        ]
    ].head(25)
)


## 18. Save final training metrics


In [ ]:
final_metrics = {
    "best_validation_macro_f1": float(best_score),
    "test_loss": float(test_metrics["loss"]),
    "test_accuracy": float(test_metrics["accuracy"]),
    "test_balanced_accuracy": float(test_metrics["balanced_accuracy"]),
    "test_macro_f1": float(test_metrics["macro_f1"]),
    "per_class": per_class.to_dict(orient="records"),
    "class_names": CLASS_NAMES,
    "class_to_id": CLASS_TO_ID,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_completed": len(history["train_loss"]),
    "pretrained_used": pretrained_used,
    "best_checkpoint": str(BEST_MODEL),
    "last_checkpoint": str(LAST_MODEL),
    "confusion_matrix_csv": str(CONFUSION_CSV),
}

with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=2)

print(json.dumps(final_metrics, indent=2))
print("\\nSaved:", METRICS_JSON)


# Notebook 12 complete

## Main outputs

```text
models/
└── condition/
    ├── condition_resnet18_best.pth
    └── condition_resnet18_last.pth

results/
└── condition/
    ├── training_history.json
    ├── condition_training_metrics.json
    ├── condition_confusion_matrix.csv
    └── incorrect_test_predictions.csv
```

### Model

**ResNet-18 → 5-class condition classifier**

```text
Day
Night
Rain
Fog
Unknown
```

The best checkpoint is selected using **validation macro-F1**, then evaluated once on the untouched test split.

### Next notebook

`13_condition_analysis_evaluation.ipynb` (if you keep this existing evaluation notebook; run it after training).


## Next notebooks in the project workflow

1. **Next — Condition-classifier evaluation:** run the existing `13_condition_analysis_evaluation.ipynb` against the saved best checkpoint and verify per-class results.
2. **Adaptive enhancement notebooks:** continue with the existing night/rain/fog enhancement notebooks in your project; do not rename or overwrite them based only on this suggested sequence. load the best condition checkpoint; define separate night enhancement and fog dehazing branches, plus a rain-restoration branch. Keep the original frame available for baseline comparisons. Do not train on the test split.
3. **YOLO Object Detection Training/Evaluation:** use the existing `dataset.yaml` and YOLO bounding-box labels; train a lightweight detector and evaluate mAP50, mAP50-95, precision, and recall by condition.
4. **Video Tracking and Risk Pipeline:** integrate frame-by-frame condition prediction, adaptive enhancement, YOLO detection, ByteTrack, approximate distance, and risk alerts. Treat distance as approximate unless calibrated ground truth is available.
5. **Comparative Experiments:** compare original video → YOLO against fixed enhancement → YOLO and condition-adaptive enhancement → YOLO; report condition-wise metrics, FPS, and latency.

These are the recommended next stages based on the project workflow. They are not executed by this notebook.